In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import joblib

In [2]:
os.makedirs("../models", exist_ok=True)
print("Models folder created successfully.")

Models folder created successfully.


In [3]:
DATA_PATH = "../data/processed/final_processed_data.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

df.head()

Dataset loaded successfully.
Shape: (52930, 25)


,TransactionId,UserId,VisitYear,VisitMonth,VisitMode,AttractionId,Rating,ContinentId,RegionId,CountryId,...,AttractionType,VisitModeId,CityId_city,CityName,CountryId_city,Country,RegionId_country,Region,ContinentId_region,Continent
0,3,70456,2022,10,2,640,5,5,21,163,...,Nature & Wildlife Areas,NaN,1,Douala,1,United Kingdom,21,Western Europe,5,Europe
1,8,7567,2022,10,4,640,5,2,8,48,...,Nature & Wildlife Areas,NaN,1,Douala,1,Canada,8,Northern America,2,America
2,9,79069,2022,10,3,640,5,2,9,54,...,Nature & Wildlife Areas,NaN,1,Douala,1,Brazil,9,South America,2,America
3,10,31019,2022,10,3,640,3,5,17,135,...,Nature & Wildlife Areas,NaN,1,Douala,1,Switzerland,17,Central Europe,5,Europe
4,15,43611,2022,10,2,640,3,5,21,163,...,Nature & Wildlife Areas,NaN,1,Douala,1,United Kingdom,21,Western Europe,5,Europe


In [4]:
print("Available Columns:")
print(df.columns.tolist())

Available Columns:
['TransactionId', 'UserId', 'VisitYear', 'VisitMonth', 'VisitMode', 'AttractionId', 'Rating', 'ContinentId', 'RegionId', 'CountryId', 'CityId', 'AttractionCityId', 'AttractionTypeId', 'Attraction', 'AttractionAddress', 'AttractionType', 'VisitModeId', 'CityId_city', 'CityName', 'CountryId_city', 'Country', 'RegionId_country', 'Region', 'ContinentId_region', 'Continent']


In [5]:
TARGET = "VisitMode"

if TARGET not in df.columns:
    raise ValueError(f"{TARGET} column not found in dataset.")

print("Target variable:", TARGET)
print("Missing values in target:", df[TARGET].isnull().sum())

Target variable: VisitMode
Missing values in target: 0


In [6]:
df = df.dropna(subset=[TARGET]).copy()

print("Dataset shape after removing missing target values:", df.shape)

Dataset shape after removing missing target values: (52930, 25)


In [7]:
label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(df[TARGET])

print("Target classes:")
print(label_encoder.classes_)
print("Number of classes:", len(label_encoder.classes_))

Target classes:
[1 2 3 4 5]
Number of classes: 5


In [8]:
drop_columns = [
    TARGET,
    "TransactionId",
    "UserId",
    "AttractionId"
]

available_drop_columns = [col for col in drop_columns if col in df.columns]

feature_columns = [
    col for col in df.columns
    if col not in available_drop_columns
]

print("Number of selected features:", len(feature_columns))
print("Selected features:")
print(feature_columns)

Number of selected features: 21
Selected features:
['VisitYear', 'VisitMonth', 'Rating', 'ContinentId', 'RegionId', 'CountryId', 'CityId', 'AttractionCityId', 'AttractionTypeId', 'Attraction', 'AttractionAddress', 'AttractionType', 'VisitModeId', 'CityId_city', 'CityName', 'CountryId_city', 'Country', 'RegionId_country', 'Region', 'ContinentId_region', 'Continent']


In [9]:
X = df[feature_columns]
y = y_encoded

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (52930, 21)
y shape: (52930,)


In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,      # 20% test, 80% train
    random_state=42,
    stratify=y
)

print("Training feature shape:", X_train.shape)
print("Testing feature shape :", X_test.shape)
print("Training target shape :", y_train.shape)
print("Testing target shape  :", y_test.shape)

print("\nTrain percentage:", round(len(X_train) / len(X) * 100, 2), "%")
print("Test percentage :", round(len(X_test) / len(X) * 100, 2), "%")

Training feature shape: (42344, 21)
Testing feature shape : (10586, 21)
Training target shape : (42344,)
Testing target shape  : (10586,)

Train percentage: 80.0 %
Test percentage : 20.0 %


In [11]:
numerical_columns = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_columns = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Numerical Columns:")
print(numerical_columns)

print("\nCategorical Columns:")
print(categorical_columns)

Numerical Columns:
['VisitYear', 'VisitMonth', 'Rating', 'ContinentId', 'RegionId', 'CountryId', 'CityId', 'AttractionCityId', 'AttractionTypeId', 'VisitModeId', 'CityId_city', 'CountryId_city', 'RegionId_country', 'ContinentId_region']

Categorical Columns:
['Attraction', 'AttractionAddress', 'AttractionType', 'CityName', 'Country', 'Region', 'Continent']


In [12]:
numerical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

print("Numerical pipeline created.")

Numerical pipeline created.


In [13]:
categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

print("Categorical pipeline created.")

Categorical pipeline created.


In [14]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_pipeline, numerical_columns),
        ("cat", categorical_pipeline, categorical_columns)
    ]
)

print("Preprocessor created successfully.")

Preprocessor created successfully.


In [15]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=50,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    ),

    "XGBoost": XGBClassifier(
        n_estimators=50,
        max_depth=4,
        learning_rate=0.1,
        random_state=42,
        n_jobs=-1,
        eval_metric="mlogloss"
    )
}

print("Models defined successfully.")
print(list(models.keys()))

Models defined successfully.
['Logistic Regression', 'Random Forest', 'XGBoost']


In [16]:
results = []
trained_models = {}

for model_name, model in models.items():
    print("\n" + "=" * 60)
    print(f"Training {model_name}")

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ]
    )

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(
        y_test,
        y_pred,
        average="weighted",
        zero_division=0
    )
    recall = recall_score(
        y_test,
        y_pred,
        average="weighted",
        zero_division=0
    )
    f1 = f1_score(
        y_test,
        y_pred,
        average="weighted",
        zero_division=0
    )

    results.append({
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1
    })

    trained_models[model_name] = pipeline

    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1 Score : {f1:.4f}")


Training Logistic Regression
Accuracy : 0.4682
Precision: 0.4246
Recall   : 0.4682
F1 Score : 0.4079

Training Random Forest
Accuracy : 0.4790
Precision: 0.4898
Recall   : 0.4790
F1 Score : 0.4166

Training XGBoost
Accuracy : 0.4739
Precision: 0.4375
Recall   : 0.4739
F1 Score : 0.4114


In [17]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="F1 Score",
    ascending=False
).reset_index(drop=True)

results_df

,Model,Accuracy,Precision,Recall,F1 Score
0,Random Forest,0.479029,0.489841,0.479029,0.416553
1,XGBoost,0.473928,0.437521,0.473928,0.411403
2,Logistic Regression,0.468166,0.424550,0.468166,0.407917


In [18]:
best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]

print("Best Model:", best_model_name)

Best Model: Random Forest


In [19]:
joblib.dump(best_model, "../models/visit_mode_model.pkl")
joblib.dump(preprocessor, "../models/visit_mode_preprocessor.pkl")
joblib.dump(label_encoder, "../models/visit_mode_label_encoder.pkl")

print("Saved successfully:")
print("- ../models/visit_mode_model.pkl")
print("- ../models/visit_mode_preprocessor.pkl")
print("- ../models/visit_mode_label_encoder.pkl")

Saved successfully:
- ../models/visit_mode_model.pkl
- ../models/visit_mode_preprocessor.pkl
- ../models/visit_mode_label_encoder.pkl


In [20]:
sample_predictions_encoded = best_model.predict(X_test.iloc[:10])
sample_predictions = label_encoder.inverse_transform(
    sample_predictions_encoded
)

actual_labels = label_encoder.inverse_transform(
    y_test[:10]
)

prediction_df = pd.DataFrame({
    "Actual Visit Mode": actual_labels,
    "Predicted Visit Mode": sample_predictions
})

prediction_df

,Actual Visit Mode,Predicted Visit Mode
0,4,3
1,3,3
2,4,2
3,2,2
4,5,2
5,2,2
6,2,2
7,2,2
8,3,3
9,5,2


In [21]:
all_predictions = best_model.predict(X_test)

comparison_df = pd.DataFrame({
    "Actual": label_encoder.inverse_transform(y_test),
    "Predicted": label_encoder.inverse_transform(all_predictions)
})

comparison_df.head(20)

,Actual,Predicted
0,4,3
1,3,3
2,4,2
3,2,2
4,5,2
5,2,2
6,2,2
7,2,2
8,3,3
9,5,2


In [22]:
# Generate predictions on the full test set
all_predictions = best_model.predict(X_test)

# Convert class names to strings
class_names = [str(cls) for cls in label_encoder.classes_]

# Print classification report
print(
    classification_report(
        y_test,
        all_predictions,
        labels=np.arange(len(class_names)),
        target_names=class_names,
        zero_division=0
    )
)

              precision    recall  f1-score   support

           1       0.00      0.00      0.00       125
           2       0.47      0.86      0.61      4324
           3       0.56      0.32      0.40      3043
           4       0.38      0.17      0.24      2189
           5       0.67      0.01      0.03       905

    accuracy                           0.48     10586
   macro avg       0.42      0.27      0.26     10586
weighted avg       0.49      0.48      0.42     10586



In [23]:
print("=" * 60)
print("Classification Modeling Completed Successfully")
print("=" * 60)

print(f"Best Model    : {best_model_name}")
print(f"Best F1 Score : {results_df.iloc[0]['F1 Score']:.4f}")
print(f"Best Accuracy : {results_df.iloc[0]['Accuracy']:.4f}")

print("\nSaved Files:")
print("1. ../models/visit_mode_model.pkl")
print("2. ../models/visit_mode_preprocessor.pkl")
print("3. ../models/visit_mode_label_encoder.pkl")

print("\nThe classification model is ready for use in the Streamlit application.")

Classification Modeling Completed Successfully
Best Model    : Random Forest
Best F1 Score : 0.4166
Best Accuracy : 0.4790

Saved Files:
1. ../models/visit_mode_model.pkl
2. ../models/visit_mode_preprocessor.pkl
3. ../models/visit_mode_label_encoder.pkl

The classification model is ready for use in the Streamlit application.


In [24]:
import json

# Save classification metrics for Streamlit
classification_metrics = {
    "Best Model": best_model_name,
    "Accuracy": float(results_df.iloc[0]["Accuracy"]),
    "Precision": float(results_df.iloc[0]["Precision"]),
    "Recall": float(results_df.iloc[0]["Recall"]),
    "F1 Score": float(results_df.iloc[0]["F1 Score"])
}

with open("../models/classification_metrics.json", "w") as f:
    json.dump(classification_metrics, f, indent=4)

print("classification_metrics.json saved successfully.")
print(classification_metrics)

classification_metrics.json saved successfully.
{'Best Model': 'Random Forest', 'Accuracy': 0.4790289061023994, 'Precision': 0.489841135507785, 'Recall': 0.4790289061023994, 'F1 Score': 0.4165528891804294}


In [25]:
import json
import pandas as pd

# Load regression metrics
with open("../models/regression_metrics.json", "r") as f:
    reg_metrics = json.load(f)

# Load classification metrics
with open("../models/classification_metrics.json", "r") as f:
    cls_metrics = json.load(f)

# Create model comparison DataFrame
model_comparison = pd.DataFrame([
    {
        "Task": "Regression",
        "Best Model": reg_metrics["Best Model"],
        "Primary Metric": "R2 Score",
        "Score": reg_metrics["R2 Score"]
    },
    {
        "Task": "Classification",
        "Best Model": cls_metrics["Best Model"],
        "Primary Metric": "F1 Score",
        "Score": cls_metrics["F1 Score"]
    }
])

# Save CSV
model_comparison.to_csv("../models/model_comparison.csv", index=False)

print("model_comparison.csv saved successfully.")
model_comparison

model_comparison.csv saved successfully.


,Task,Best Model,Primary Metric,Score
0,Regression,XGBoost,R2 Score,0.125927
1,Classification,Random Forest,F1 Score,0.416553
